<a href="https://colab.research.google.com/github/n1lays1ngh/SoulTune/blob/main/SongSoul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Download the FMA metadata zip file
!wget https://os.unil.cloud.switch.ch/fma/fma_metadata.zip

# Unzip the contents (suppressing the output to keep the notebook clean)
!unzip -q -o fma_metadata.zip
print("FMA Metadata successfully downloaded and extracted!")

--2026-05-26 20:14:17--  https://os.unil.cloud.switch.ch/fma/fma_metadata.zip
Resolving os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)... 86.119.28.16, 2001:620:5ca1:201::214
Connecting to os.unil.cloud.switch.ch (os.unil.cloud.switch.ch)|86.119.28.16|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 358412441 (342M) [application/zip]
Saving to: ‘fma_metadata.zip’

fma_metadata.zip    100%[===================>] 341.81M  19.5MB/s    in 18s     

2026-05-26 20:14:37 (18.7 MB/s) - ‘fma_metadata.zip’ saved [358412441/358412441]

FMA Metadata successfully downloaded and extracted!


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib
import warnings

warnings.filterwarnings('ignore')
print("Loading CSVs... (This might take 30-60 seconds due to file size)")

# Load the features and tracks dataframes with their multi-level headers
features = pd.read_csv('fma_metadata/features.csv', index_col=0, header=[0, 1, 2])
tracks = pd.read_csv('fma_metadata/tracks.csv', index_col=0, header=[0, 1])

print(f"Loaded Features Shape: {features.shape}")
print(f"Loaded Tracks Shape: {tracks.shape}")

Loading CSVs... (This might take 30-60 seconds due to file size)
Loaded Features Shape: (106574, 518)
Loaded Tracks Shape: (106574, 52)


In [ ]:
# 1. Filter for the 'small' subset (8,000 tracks, perfectly balanced across 8 genres)
small_subset = tracks['set', 'subset'] == 'small'
tracks_small = tracks[small_subset]

# 2. Extract the target label (y)
y = tracks_small['track', 'genre_top']

# 3. Extract the corresponding features for the small subset
features_small = features[small_subset]

# To avoid messy column names, we isolate only the 'mean' statistics from Librosa
mean_features = features_small.xs('mean', level=1, axis=1)

# Select only the features we need for our Sonic DNA (MFCCs, Spectral Centroid, RMSE)
feature_groups = ['mfcc', 'spectral_centroid', 'rmse']
X = mean_features.loc[:, mean_features.columns.get_level_values(0).isin(feature_groups)]

# Drop any potential NaNs just to be safe
valid_indices = y.notna() & X.notna().all(axis=1)
X = X[valid_indices]
y = y[valid_indices]

print(f"Final X (Features) Shape: {X.shape}")
print(f"Final y (Labels) Shape: {y.shape}")
print("\nUnique Genres Found:", y.unique().tolist())

Final X (Features) Shape: (8000, 22)
Final y (Labels) Shape: (8000,)

Unique Genres Found: ['Hip-Hop', 'Pop', 'Folk', 'Experimental', 'Rock', 'International', 'Electronic', 'Instrumental']


In [ ]:
# 1. Encode the string labels (e.g., 'Electronic' -> 1) into numbers
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

# 2. Train / Test Split (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# 3. Scale the features (Standardization)
# We MUST save this scaler later, because incoming audio in Phase 2 needs to be scaled exactly the same way!
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Train the Random Forest Classifier
print("Training Random Forest Classifier... 🌲")
clf = RandomForestClassifier(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
clf.fit(X_train_scaled, y_train)

print("Training Complete!")

Training Random Forest Classifier... 🌲
Training Complete!


In [ ]:
# 1. Predict on the test set
y_pred = clf.predict(X_test_scaled)

# 2. Print the Baseline Scores
accuracy = accuracy_score(y_test, y_pred)
print(f"Classification Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")
# Convert numbers back to genre names for the report
target_names = label_encoder.inverse_transform(np.unique(y_test))
print(classification_report(y_test, y_pred, target_names=target_names))

# 3. Save the Model, Scaler, and Label Encoder for Phase 2
joblib.dump(clf, 'sonic_dna_rf_model.pkl')
joblib.dump(scaler, 'sonic_dna_scaler.pkl')
joblib.dump(label_encoder, 'sonic_dna_label_encoder.pkl')

print("✅ Saved sonic_dna_rf_model.pkl, sonic_dna_scaler.pkl, and sonic_dna_label_encoder.pkl successfully.")

Classification Accuracy: 45.25%

Detailed Classification Report:
               precision    recall  f1-score   support

   Electronic       0.42      0.48      0.45       184
 Experimental       0.44      0.32      0.37       200
         Folk       0.46      0.58      0.51       182
      Hip-Hop       0.45      0.49      0.47       221
 Instrumental       0.48      0.53      0.50       194
International       0.50      0.44      0.47       213
          Pop       0.23      0.16      0.19       196
         Rock       0.56      0.62      0.59       210

     accuracy                           0.45      1600
    macro avg       0.44      0.45      0.44      1600
 weighted avg       0.44      0.45      0.44      1600

✅ Saved sonic_dna_rf_model.pkl, sonic_dna_scaler.pkl, and sonic_dna_label_encoder.pkl successfully.


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. Train / Test Split (already done in memory from Cell 3/4, but redefining to be safe)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# 2. Scale the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 3. Train the XGBoost Classifier
print("Training XGBoost Classifier... 🚀")
# We use standard tuning parameters for multi-class classification
xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=8,
    n_estimators=250,        # More trees than RF
    learning_rate=0.05,      # Slower learning rate to prevent overfitting
    max_depth=6,             # Shallower trees
    subsample=0.8,           # Use 80% of data per tree
    colsample_bytree=0.8,    # Use 80% of features per tree
    random_state=42,
    n_jobs=-1
)

xgb_clf.fit(X_train_scaled, y_train)
print("XGBoost Training Complete!")

Training XGBoost Classifier... 🚀
XGBoost Training Complete!


In [ ]:
# 1. Predict on the test set
y_pred = xgb_clf.predict(X_test_scaled)

# 2. Print the Baseline Scores
accuracy = accuracy_score(y_test, y_pred)
print(f"XGBoost Classification Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")

# Convert numbers back to genre names for the report
target_names = label_encoder.inverse_transform(np.unique(y_test))
print(classification_report(y_test, y_pred, target_names=target_names))

# 3. Save the Model, Scaler, and Label Encoder for Phase 2
joblib.dump(xgb_clf, 'sonic_dna_xgb_model2.pkl')
joblib.dump(scaler, 'sonic_dna_scaler2.pkl')
joblib.dump(label_encoder, 'sonic_dna_label_encoder2.pkl')

print("✅ Saved sonic_dna_xgb_model2.pkl, sonic_dna_scaler2.pkl, and sonic_dna_label_encoder2.pkl successfully.")

XGBoost Classification Accuracy: 48.25%

Detailed Classification Report:
               precision    recall  f1-score   support

   Electronic       0.47      0.51      0.49       184
 Experimental       0.41      0.34      0.38       200
         Folk       0.52      0.62      0.56       182
      Hip-Hop       0.49      0.52      0.50       221
 Instrumental       0.51      0.55      0.53       194
International       0.50      0.46      0.48       213
          Pop       0.31      0.24      0.27       196
         Rock       0.58      0.61      0.60       210

     accuracy                           0.48      1600
    macro avg       0.47      0.48      0.48      1600
 weighted avg       0.48      0.48      0.48      1600

✅ Saved sonic_dna_xgb_model2.pkl, sonic_dna_scaler2.pkl, and sonic_dna_label_encoder2.pkl successfully.


In [ ]:
# 1. Filter for the 'small' subset (8,000 tracks)
small_subset = tracks['set', 'subset'] == 'small'
y = tracks.loc[small_subset, ('track', 'genre_top')]

# 2. Extract ALL 518 features for the small subset
X = features.loc[small_subset]

# 3. Clean the data (StandardScaler hates missing values, so we fill NaNs with 0)
X = X.fillna(0)

# Drop any rows where the target label is missing
valid_indices = y.notna()
X = X[valid_indices]
y = y[valid_indices]

print(f"Full X (Features) Shape: {X.shape} 🚀")
print(f"Final y (Labels) Shape: {y.shape}")

Full X (Features) Shape: (8000, 518) 🚀
Final y (Labels) Shape: (8000,)


In [ ]:
import xgboost as xgb
from sklearn.metrics import accuracy_score, classification_report
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from tqdm.auto import tqdm

# --- 1. Custom Progress Bar Callback ---
class TqdmCallback(xgb.callback.TrainingCallback):
    def __init__(self, total_trees):
        self.pbar = tqdm(total=total_trees, desc="Training XGBoost Trees")

    def after_iteration(self, model, epoch, evals_log):
        self.pbar.update(1)
        return False # Return False to continue training

    def __del__(self):
        self.pbar.close()

# --- 2. Encode and Split ---
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

# --- 3. Scale the features ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# --- 4. Train the XGBoost Classifier ---
n_trees = 300
print("Initializing Full-Feature XGBoost Classifier... 🚀")

xgb_clf = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=8,
    n_estimators=n_trees,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    device='cuda',           # <--- ADD THIS LINE!
    random_state=42,
    n_jobs=-1,
    callbacks=[TqdmCallback(total_trees=n_trees)]
)

# Fit without the callback argument
xgb_clf.fit(X_train_scaled, y_train)

print("XGBoost Training Complete!")

Initializing Full-Feature XGBoost Classifier... 🚀


Training XGBoost Trees:   0%|          | 0/300 [00:00<?, ?it/s]

XGBoost Training Complete!


In [ ]:
# 1. Predict on the test set
y_pred = xgb_clf.predict(X_test_scaled)

# 2. Print the Scores
accuracy = accuracy_score(y_test, y_pred)
print(f"Full-Feature XGBoost Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Classification Report:")

# Convert numbers back to genre names for the report
target_names = label_encoder.inverse_transform(np.unique(y_test))
print(classification_report(y_test, y_pred, target_names=target_names))

# --- THE FIX: Remove the un-picklable progress bar callback ---
xgb_clf.set_params(callbacks=None)

# 3. Save the Model, Scaler, and Encoder
joblib.dump(xgb_clf, 'sonic_dna_xgb_full.pkl')
joblib.dump(scaler, 'sonic_dna_scaler_full.pkl')
joblib.dump(label_encoder, 'sonic_dna_label_encoder_full.pkl')

print("✅ Saved sonic_dna_xgb_full.pkl, sonic_dna_scaler_full.pkl, and sonic_dna_label_encoder_full.pkl successfully.")

Full-Feature XGBoost Accuracy: 60.25%

Detailed Classification Report:
               precision    recall  f1-score   support

   Electronic       0.56      0.59      0.58       184
 Experimental       0.53      0.48      0.51       200
         Folk       0.66      0.68      0.67       182
      Hip-Hop       0.69      0.68      0.69       221
 Instrumental       0.60      0.67      0.63       194
International       0.73      0.64      0.69       213
          Pop       0.38      0.39      0.38       196
         Rock       0.66      0.68      0.67       210

     accuracy                           0.60      1600
    macro avg       0.60      0.60      0.60      1600
 weighted avg       0.60      0.60      0.60      1600

✅ Saved sonic_dna_xgb_full.pkl, sonic_dna_scaler_full.pkl, and sonic_dna_label_encoder_full.pkl successfully.


In [ ]:
!pip install -q optuna
import optuna
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import xgboost as xgb
import joblib

print("Optuna installed and ready!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 17.2 MB/s eta 0:00:00
Optuna installed and ready!


In [ ]:
# Create a dedicated validation set just for Optuna to test its guesses against
# We split our existing training data so we don't touch the final hold-out test set
X_opt_train, X_opt_val, y_opt_train, y_opt_val = train_test_split(
    X_train_scaled, y_train, test_size=0.2, random_state=42
)

def objective(trial):
    # 1. Define the hyperparameter search space
    param = {
        'objective': 'multi:softprob',
        'num_class': 8,
        'tree_method': 'hist',
        'device': 'cuda',          # CRITICAL: Forces Optuna to use the GPU for speed
        'n_estimators': trial.suggest_int('n_estimators', 200, 500),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 7),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True)
    }

    # 2. Build the model for this specific trial
    opt_model = xgb.XGBClassifier(**param, random_state=42)

    # 3. Train it
    opt_model.fit(X_opt_train, y_opt_train, verbose=False)

    # 4. Predict and evaluate
    preds = opt_model.predict(X_opt_val)
    accuracy = accuracy_score(y_opt_val, preds)

    # Return the accuracy so Optuna knows how well it did
    return accuracy

# Create the study and tell it to MAXIMIZE accuracy
print("Starting Optuna Hyperparameter Search... 🚀")
study = optuna.create_study(direction='maximize')
# Running 30 trials. On a GPU, this should take about 3 to 5 minutes.
study.optimize(objective, n_trials=30)

print("\n🏆 Best Trial Found!")
print(f"Validation Accuracy: {study.best_trial.value * 100:.2f}%")
print("Best Parameters:", study.best_trial.params)

[I 2026-05-26 20:47:26,000] A new study created in memory with name: no-name-87cd30c3-c3fa-4671-8c5f-931b7bf8cc44


Starting Optuna Hyperparameter Search... 🚀


[I 2026-05-26 20:47:46,905] Trial 0 finished with value: 0.59765625 and parameters: {'n_estimators': 302, 'max_depth': 5, 'learning_rate': 0.07169981386730499, 'subsample': 0.9824631283659777, 'colsample_bytree': 0.7167188700536082, 'min_child_weight': 3, 'gamma': 0.00010243994267768049}. Best is trial 0 with value: 0.59765625.
[I 2026-05-26 20:48:34,276] Trial 1 finished with value: 0.59765625 and parameters: {'n_estimators': 375, 'max_depth': 12, 'learning_rate': 0.01888706237536996, 'subsample': 0.6328109846619048, 'colsample_bytree': 0.6523524416068931, 'min_child_weight': 7, 'gamma': 0.006681999496423528}. Best is trial 0 with value: 0.59765625.
[I 2026-05-26 20:48:45,899] Trial 2 finished with value: 0.56484375 and parameters: {'n_estimators': 278, 'max_depth': 5, 'learning_rate': 0.015166384411853463, 'subsample': 0.7683155327134974, 'colsample_bytree': 0.5908650319546733, 'min_child_weight': 1, 'gamma': 4.454375154236251e-06}. Best is trial 0 with value: 0.59765625.
[I 2026-05-


🏆 Best Trial Found!
Validation Accuracy: 61.88%
Best Parameters: {'n_estimators': 466, 'max_depth': 9, 'learning_rate': 0.05997399385483124, 'subsample': 0.9399720942548245, 'colsample_bytree': 0.7049347394521754, 'min_child_weight': 4, 'gamma': 9.196136252294279e-07}


In [ ]:
# 1. Grab the best parameters discovered by Optuna
best_params = study.best_trial.params

# Add back our required baseline settings
best_params['objective'] = 'multi:softprob'
best_params['num_class'] = 8
best_params['tree_method'] = 'hist'
best_params['random_state'] = 42
# We can remove 'device': 'cuda' here if you want the final saved model
# to be strictly CPU-compatible for your API later, but training it on GPU now is fine.
best_params['device'] = 'cuda'

# 2. Initialize the Champion Model
print("Training the Champion XGBoost Model with Optuna's parameters... 🥇")
champion_clf = xgb.XGBClassifier(**best_params)

# 3. Train on the FULL training set
champion_clf.fit(X_train_scaled, y_train)

# 4. Final Evaluation on the true Hold-Out Test Set
final_preds = champion_clf.predict(X_test_scaled)
final_accuracy = accuracy_score(y_test, final_preds)

print(f"\nFinal Champion Accuracy: {final_accuracy * 100:.2f}% 🚀\n")

from sklearn.metrics import classification_report
target_names = label_encoder.inverse_transform(np.unique(y_test))
print(classification_report(y_test, final_preds, target_names=target_names))

# 5. Save the Champion Model (overwriting the old one)
import joblib
champion_clf.set_params(device='cpu') # Ensure it saves ready for a CPU API
joblib.dump(champion_clf, 'sonic_dna_xgb_full_3838.pkl')
print("✅ Saved the Champion Model successfully.")

Training the Champion XGBoost Model with Optuna's parameters... 🥇

Final Champion Accuracy: 60.94% 🚀

               precision    recall  f1-score   support

   Electronic       0.58      0.61      0.60       184
 Experimental       0.53      0.51      0.52       200
         Folk       0.66      0.69      0.67       182
      Hip-Hop       0.69      0.67      0.68       221
 Instrumental       0.60      0.65      0.63       194
International       0.71      0.65      0.68       213
          Pop       0.40      0.37      0.38       196
         Rock       0.66      0.70      0.68       210

     accuracy                           0.61      1600
    macro avg       0.61      0.61      0.61      1600
 weighted avg       0.61      0.61      0.61      1600

✅ Saved the Champion Model successfully.
